# GRPO: Improve DataExtractor Quality on Qwen3.5-0.8B

Use Group Relative Policy Optimization to teach the SFT-tuned model better data extraction.

**Starting point:** Format SFT LoRA (99.3% format compliance) — continue training same adapter.

**Training data:** `grpo_extractor_mutated.jsonl` — 41,767 examples with ground truth field_ids + field_values.

**Reward signals:**
1. `format_reward` — DSPy `[[ ## field ## ]]` structure intact (+2.0)
2. `accuracy_reward` — F1 on field_ids + exact match on field_values (+3.0)
3. `hallucination_penalty` — penalize extracting fields not in form schema (-2.0)

**Approach:** Vanilla HuggingFace + TRL + PEFT (no Unsloth). Qwen3.5's hybrid VLM architecture
has known bugs with Unsloth's compiled code path ([#4612](https://github.com/unslothai/unsloth/issues/4612),
[#4801](https://github.com/unslothai/unsloth/issues/4801)). Vanilla HF + TRL works perfectly —
verified on CPU locally.

## 1. Install Dependencies

No Unsloth needed — vanilla HuggingFace + TRL + PEFT with BitsAndBytes QLoRA.

In [ ]:
%%capture
!pip install --upgrade -qqq pip
!pip install -qqq \
    "transformers==5.5.4" \
    "trl==1.1.0" \
    "peft==0.19.0" \
    "datasets>=4.0" \
    "accelerate==1.13.0" \
    "bitsandbytes>=0.45" \
    "safetensors"

# Optional: faster Qwen3.5 attention (Gated DeltaNet layers)
# Falls back to torch implementation if install fails — that's fine
!pip install -qqq --no-build-isolation flash-linear-attention causal_conv1d==1.6.0 2>/dev/null || true

import torch, transformers, trl, peft
print(f"torch:        {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")
print(f"peft:         {peft.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Drive + Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/qwen35_08b_finetuning"
SFT_LORA_PATH = f"{BASE_DIR}/qwen35-08b-dspy-format-lora"
DATA_PATH = f"{BASE_DIR}/grpo_extractor_mutated.jsonl"
OUTPUT_DIR = f"{BASE_DIR}/grpo_outputs"

for p in [SFT_LORA_PATH, DATA_PATH]:
    assert os.path.exists(p), f"Missing: {p}"
print("All paths verified")

## 3. Load Base Model + SFT LoRA

Load Qwen3.5-0.8B with 4-bit QLoRA quantization, then load SFT LoRA adapter on top.

**Important:** The SFT LoRA was trained on `Qwen3_5ForConditionalGeneration` (VLM model class)
via Unsloth's FastVisionModel. We must load the same class here — `AutoModelForCausalLM` loads
`Qwen3_5ForCausalLM` which has different layer nesting, causing the LoRA adapter keys to mismatch
and silently fail to load.

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig
from transformers.models.qwen3_5.modeling_qwen3_5 import Qwen3_5ForConditionalGeneration
from peft import PeftModel
import torch

# QLoRA config — fits Qwen3.5-0.8B comfortably on T4 (16GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model — MUST use Qwen3_5ForConditionalGeneration (not AutoModelForCausalLM)
# because the SFT LoRA was trained on this class via Unsloth's FastVisionModel.
# AutoModelForCausalLM loads Qwen3_5ForCausalLM which has different layer nesting
# (model.layers.X vs model.model.layers.X), causing LoRA adapter keys to silently mismatch.
BASE_MODEL = "Qwen/Qwen3.5-0.8B"
print(f"Loading base model: {BASE_MODEL}")
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

# Load tokenizer from SFT LoRA (has the right special tokens)
tokenizer = AutoTokenizer.from_pretrained(SFT_LORA_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load SFT LoRA adapter — is_trainable=True to continue training with GRPO
print(f"Loading SFT LoRA: {SFT_LORA_PATH}")
model = PeftModel.from_pretrained(model, SFT_LORA_PATH, is_trainable=True)

# Enable gradient checkpointing to save memory during training
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total:,} total params, {trainable:,} trainable ({trainable/total*100:.1f}%)")

# Verify LoRA adapter is actually loaded (not random)
import numpy as np
lora_stds = []
for name, param in model.named_parameters():
    if "lora_B" in name and param.requires_grad:
        lora_stds.append(param.data.std().item())
avg_std = np.mean(lora_stds) if lora_stds else 0
print(f"LoRA B avg std: {avg_std:.6f} (should be >0 if SFT weights loaded, ~0 if random init)")

## 4. Load Training Data

No VLM format conversion needed — vanilla HF uses plain string `content` in messages.

In [ ]:
import json
from collections import Counter
from datasets import Dataset

with open(DATA_PATH) as f:
    raw_data = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw_data)} examples")
print(f"Source distribution: {Counter(r['source'] for r in raw_data)}")
print(f"Avg fields per example: {sum(len(r['ground_truth_ids']) for r in raw_data) / len(raw_data):.1f}")

# Convert to dataset — prompts are already [{role, content}] message lists
# No VLM typed-block conversion needed with vanilla HF
train_data = []
for r in raw_data:
    train_data.append({
        "prompt": r["prompt"],  # [{role: "system", content: "..."}, {role: "user", content: "..."}]
        "ground_truth_ids": json.dumps(r["ground_truth_ids"]),
        "ground_truth_values": json.dumps(r["ground_truth_values"]),
    })

train_dataset = Dataset.from_list(train_data)
print(f"\nDataset: {len(train_dataset)} examples")
print(f"Columns: {train_dataset.column_names}")
print(f"Sample prompt roles: {[m['role'] for m in train_dataset[0]['prompt']]}")

## 5. Reward Functions

In [ ]:
import re
import json
import ast


def extract_completion_text(completion) -> str:
    """Extract text from a completion, handling various formats."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and len(completion) > 0:
        msg = completion[0]
        if isinstance(msg, dict):
            content = msg.get("content", "")
            if isinstance(content, str):
                return content
            if isinstance(content, list):
                return " ".join(c.get("text", "") for c in content if isinstance(c, dict))
    return str(completion)


def extract_prompt_text(prompt) -> str:
    """Extract all text from a prompt."""
    if isinstance(prompt, str):
        return prompt
    if isinstance(prompt, list):
        parts = []
        for msg in prompt:
            if isinstance(msg, dict):
                content = msg.get("content", "")
                if isinstance(content, str):
                    parts.append(content)
                elif isinstance(content, list):
                    parts.append(" ".join(c.get("text", "") for c in content if isinstance(c, dict)))
        return "\n".join(parts)
    return str(prompt)


def parse_list(s: str) -> list:
    """Parse a list string — handles both JSON double-quotes and Python single-quotes."""
    s = s.strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        pass
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return result
    except (ValueError, SyntaxError):
        pass
    return []


def parse_dspy_output(completion: str) -> tuple[list[str], list[str]]:
    """Parse field_ids and field_values from DSPy-formatted completion."""
    try:
        if "[[ ## field_ids ## ]]" not in completion or "[[ ## field_values ## ]]" not in completion:
            return [], []
        ids_part = completion.split("[[ ## field_ids ## ]]")[1].split("[[ ## field_values ## ]]")[0].strip()
        vals_raw = completion.split("[[ ## field_values ## ]]")[1]
        if "[[ ## completed ## ]]" in vals_raw:
            vals_part = vals_raw.split("[[ ## completed ## ]]")[0].strip()
        else:
            vals_part = vals_raw.strip()
        field_ids = parse_list(ids_part)
        field_values = parse_list(vals_part)
        if not isinstance(field_ids, list) or not isinstance(field_values, list):
            return [], []
        return field_ids, field_values
    except (IndexError, ValueError):
        return [], []


def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward for correct DSPy output format. Max +2.0."""
    scores = []
    for completion in completions:
        text = extract_completion_text(completion)
        score = 0.0
        has_ids = "[[ ## field_ids ## ]]" in text
        has_vals = "[[ ## field_values ## ]]" in text
        has_done = "[[ ## completed ## ]]" in text
        if has_ids and has_vals and has_done:
            score += 1.0
            field_ids, field_values = parse_dspy_output(text)
            if len(field_ids) > 0 and len(field_ids) == len(field_values):
                score += 1.0
        scores.append(score)
    return scores


def accuracy_reward_func(completions, ground_truth_ids, ground_truth_values, **kwargs) -> list[float]:
    """Reward for correct extraction. Max +3.0 (1.5 ID F1 + 1.5 value match)."""
    scores = []
    for completion, gt_ids_json, gt_vals_json in zip(completions, ground_truth_ids, ground_truth_values):
        text = extract_completion_text(completion)
        gt_ids = json.loads(gt_ids_json)
        gt_vals = json.loads(gt_vals_json)
        gt_map = dict(zip(gt_ids, gt_vals))
        pred_ids, pred_vals = parse_dspy_output(text)
        pred_map = dict(zip(pred_ids, pred_vals))
        if not gt_ids:
            scores.append(1.5 if len(pred_ids) == 0 else 0.0)
            continue
        if not pred_ids:
            scores.append(0.0)
            continue
        gt_set = set(gt_ids)
        pred_set = set(pred_ids)
        tp = len(gt_set & pred_set)
        precision = tp / len(pred_set) if pred_set else 0
        recall = tp / len(gt_set) if gt_set else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        id_score = f1 * 1.5
        correct_values = 0
        for fid in gt_set & pred_set:
            if str(pred_map.get(fid, "")).strip() == str(gt_map[fid]).strip():
                correct_values += 1
        val_score = (correct_values / len(gt_set)) * 1.5 if gt_set else 0
        scores.append(id_score + val_score)
    if completions:
        text = extract_completion_text(completions[0])
        gt = json.loads(ground_truth_ids[0])
        pred, _ = parse_dspy_output(text)
        print(f"--- GT: {gt} | Pred: {pred} | Score: {scores[0]:.2f}")
    return scores


def hallucination_penalty_func(prompts, completions, **kwargs) -> list[float]:
    """Penalize extracting field_ids not in form schema. Min -2.0."""
    scores = []
    for prompt, completion in zip(prompts, completions):
        text = extract_completion_text(completion)
        prompt_text = extract_prompt_text(prompt)
        valid_ids = set(re.findall(
            r'^\s+(\w+)\s+\((?:text|date|select|email|tel|number|textarea|checkbox|radio|file)',
            prompt_text, re.MULTILINE
        ))
        pred_ids, _ = parse_dspy_output(text)
        if not valid_ids or not pred_ids:
            scores.append(0.0)
            continue
        hallucinated = [fid for fid in pred_ids if fid not in valid_ids]
        penalty = min(len(hallucinated) * 0.5, 2.0)
        scores.append(-penalty)
    return scores


# Sanity check
test_good = '[[ ## field_ids ## ]]\n["full_name", "email"]\n[[ ## field_values ## ]]\n["Jane Smith", "jane@email.com"]\n[[ ## completed ## ]]'
test_bad = "I extracted the name field for you."
test_single = "[[ ## field_ids ## ]]\n['full_name']\n[[ ## field_values ## ]]\n['Jane']\n[[ ## completed ## ]]"
print("Good:", format_reward_func([test_good]), parse_dspy_output(test_good))
print("Bad: ", format_reward_func([test_bad]), parse_dspy_output(test_bad))
print("Single-quote:", format_reward_func([test_single]), parse_dspy_output(test_single))

## 6. GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    # Training schedule
    max_steps=300,
    save_steps=100,
    logging_steps=1,

    # Batch / generation
    num_generations=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_completion_length=512,

    # Optimizer
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    max_grad_norm=0.1,

    # Precision
    bf16=True,

    # Logging
    log_completions=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,
        accuracy_reward_func,
        hallucination_penalty_func,
    ],
    train_dataset=train_dataset,
)
print("GRPOTrainer created")

In [ ]:
# GPU memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB reserved before training.\n")

# Train!
trainer.train()

In [ ]:
# Memory stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_grpo = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for GRPO = {used_memory_for_grpo} GB.")
print(f"Peak reserved memory % of max = {used_percentage}%.")

## 7. Post-training Evaluation

In [ ]:
# Quick eval on a few examples
model.eval()

for idx in [100, 200, 500, 1000]:
    if idx >= len(train_dataset):
        continue
    sample = train_dataset[idx]
    messages = sample["prompt"]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)

    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred_ids, pred_vals = parse_dspy_output(response)
    gt_ids = json.loads(sample['ground_truth_ids'])
    gt_vals = json.loads(sample['ground_truth_values'])

    print(f"\n{'='*60}")
    print(f"Example {idx}")
    print(f"GT  IDs:  {gt_ids}")
    print(f"GT  Vals: {gt_vals}")
    print(f"Pred IDs: {pred_ids}")
    print(f"Pred Vals:{pred_vals}")
    match = pred_ids == gt_ids and pred_vals == gt_vals
    print(f"Exact match: {'Y' if match else 'N'}")

In [ ]:
# Batch eval on 200 random examples
import random
random.seed(42)
eval_indices = random.sample(range(len(train_dataset)), min(200, len(train_dataset)))

exact_matches = 0
id_f1_scores = []
format_ok = 0
total = len(eval_indices)

model.eval()
for i, idx in enumerate(eval_indices):
    sample = train_dataset[idx]
    messages = sample["prompt"]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)

    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred_ids, pred_vals = parse_dspy_output(response)
    gt_ids = json.loads(sample['ground_truth_ids'])
    gt_vals = json.loads(sample['ground_truth_values'])

    if pred_ids and len(pred_ids) == len(pred_vals):
        format_ok += 1
    if pred_ids == gt_ids and pred_vals == gt_vals:
        exact_matches += 1

    gt_set = set(gt_ids)
    pred_set = set(pred_ids)
    if gt_set or pred_set:
        tp = len(gt_set & pred_set)
        p = tp / len(pred_set) if pred_set else 0
        r = tp / len(gt_set) if gt_set else 0
        f1 = 2*p*r/(p+r) if (p+r) > 0 else 0
        id_f1_scores.append(f1)

    if (i+1) % 50 == 0:
        print(f"Evaluated {i+1}/{total}...")

print(f"\n{'='*50}")
print(f"Results on {total} examples:")
print(f"  Format compliance: {format_ok/total*100:.1f}%")
print(f"  Exact match:      {exact_matches/total*100:.1f}%")
if id_f1_scores:
    print(f"  Field ID F1:      {sum(id_f1_scores)/len(id_f1_scores)*100:.1f}%")

## 8. Save Model

In [ ]:
import shutil

# Save LoRA adapter
lora_save_path = "qwen35-08b-grpo-extractor-lora"
model.save_pretrained(lora_save_path)
tokenizer.save_pretrained(lora_save_path)
print(f"Saved LoRA adapter to {lora_save_path}")

# Verify LoRA weights are trained (not zeros)
from safetensors import safe_open
import glob as glob_mod
for f in glob_mod.glob(f"{lora_save_path}/*.safetensors"):
    with safe_open(f, framework="pt") as st:
        for key in list(st.keys())[:5]:
            tensor = st.get_tensor(key)
            print(f"  {key}: shape={tensor.shape}, mean={tensor.mean():.6f}, std={tensor.std():.6f}")

# Copy to Google Drive
drive_lora = f"{BASE_DIR}/qwen35-08b-grpo-extractor-lora"
shutil.copytree(lora_save_path, drive_lora, dirs_exist_ok=True)
print(f"\nCopied LoRA to Drive: {drive_lora}")

In [ ]:
# Merge LoRA into base model and save for GGUF conversion
print("Merging LoRA into base model...")
merged_model = model.merge_and_unload()

merged_save_path = "qwen35-08b-grpo-extractor-merged"
merged_model.save_pretrained(merged_save_path)
tokenizer.save_pretrained(merged_save_path)
print(f"Saved merged model to {merged_save_path}")

# Copy to Drive
drive_merged = f"{BASE_DIR}/qwen35-08b-grpo-extractor-merged"
shutil.copytree(merged_save_path, drive_merged, dirs_exist_ok=True)
print(f"Copied merged model to Drive: {drive_merged}")

print("\n--- GGUF Conversion (run locally) ---")
print("1. Download merged model from Google Drive")
print("2. pip install llama-cpp-python")
print("3. git clone https://github.com/ggerganov/llama.cpp")
print("4. python llama.cpp/convert_hf_to_gguf.py qwen35-08b-grpo-extractor-merged --outtype q8_0")